In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/nyc_311_2025.csv")
df.head(5)

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,status,borough,incident_zip,community_board,open_data_channel_type,latitude,longitude
0,63577994,2025-01-01T00:00:12.000,2025-01-02T17:05:54.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,Closed,BRONX,10466.0,12 BRONX,MOBILE,40.891872,-73.860168
1,63584096,2025-01-01T00:00:13.000,2025-01-01T00:05:19.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,Closed,BROOKLYN,11211.0,01 BROOKLYN,MOBILE,40.711210,-73.958973
2,63581480,2025-01-01T00:00:17.000,2025-01-01T09:01:06.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,Closed,BRONX,10466.0,12 BRONX,MOBILE,40.891872,-73.860168
3,63575612,2025-01-01T00:00:19.000,2025-01-01T00:43:08.000,NYPD,New York City Police Department,Illegal Fireworks,NaN,Street/Sidewalk,Closed,BROOKLYN,11230.0,14 BROOKLYN,ONLINE,40.629240,-73.968790
4,63586536,2025-01-01T00:00:34.000,2025-01-02T12:04:30.000,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,APARTMENT ONLY,RESIDENTIAL BUILDING,Closed,BRONX,10466.0,12 BRONX,ONLINE,40.891940,-73.852775


In [5]:
df.shape

(500000, 15)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   unique_key              500000 non-null  int64  
 1   created_date            500000 non-null  str    
 2   closed_date             495498 non-null  str    
 3   agency                  500000 non-null  str    
 4   agency_name             500000 non-null  str    
 5   complaint_type          500000 non-null  str    
 6   descriptor              491688 non-null  str    
 7   location_type           446443 non-null  str    
 8   status                  500000 non-null  str    
 9   borough                 500000 non-null  str    
 10  incident_zip            496194 non-null  float64
 11  community_board         500000 non-null  str    
 12  open_data_channel_type  500000 non-null  str    
 13  latitude                494811 non-null  float64
 14  longitude               494811 

In [7]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
unique_key,500000.0,NaN,NaN,NaN,63846838.838438,152952.964217,63572086.0,63714623.75,63845006.5,63979154.25,69074345.0
created_date,500000,411705,2025-01-15T07:00:00.000,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
closed_date,495498,300454,2025-02-19T00:00:00.000,261,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agency,500000,14,NYPD,214464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agency_name,500000,14,New York City Police Department,214464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
complaint_type,500000,170,HEAT/HOT WATER,92508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
descriptor,491688,784,Loud Music/Party,72887,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_type,446443,117,RESIDENTIAL BUILDING,155653,NaN,NaN,NaN,NaN,NaN,NaN,NaN
status,500000,7,Closed,496534,NaN,NaN,NaN,NaN,NaN,NaN,NaN
borough,500000,6,BRONX,148411,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.isnull().sum().sort_values(ascending=False)

location_type             53557
descriptor                 8312
longitude                  5189
latitude                   5189
closed_date                4502
incident_zip               3806
unique_key                    0
complaint_type                0
agency_name                   0
agency                        0
created_date                  0
borough                       0
status                        0
open_data_channel_type        0
community_board               0
dtype: int64

In [9]:
categorical_cols = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "status",
    "borough",
    "incident_zip",
    "community_board",
    "open_data_channel_type"
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print("Unique values:", df[col].nunique(dropna=True))
    print(df[col].value_counts(dropna=False).head(10))


===== agency =====
Unique values: 14
agency
NYPD     214464
HPD      155653
DSNY      39548
DOT       27438
DEP       22992
DOB       11472
DOHMH      9386
DPR        6412
TLC        4229
DHS        3275
Name: count, dtype: int64

===== agency_name =====
Unique values: 14
agency_name
New York City Police Department                       214464
Department of Housing Preservation and Development    155653
Department of Sanitation                               39548
Department of Transportation                           27438
Department of Environmental Protection                 22992
Department of Buildings                                11472
Department of Health and Mental Hygiene                 9386
Department of Parks and Recreation                      6412
Taxi and Limousine Commission                           4229
Department of Homeless Services                         3275
Name: count, dtype: int64

===== complaint_type =====
Unique values: 170
complaint_type
HEAT/HOT WATER  

In [10]:
## Calculate resolution time
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

df['resolution_time'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600  # in hours
df['resolution_time'].describe()  

count    495498.000000
mean        234.344772
std         844.950231
min        -744.016667
25%           1.796667
50%          14.934722
75%          68.591389
max       14984.766111
Name: resolution_time, dtype: float64

In [17]:
df["resolution_time"].isna().sum()

np.int64(4502)

In [18]:
df["resolution_time"].value_counts().head(10)

resolution_time
0.000000    6278
0.100000     180
0.083333     176
0.050000     175
0.016667     171
0.066667     167
0.033333     167
0.133333     164
0.116667     160
0.166667     156
Name: count, dtype: int64

Your resolution_time has:

495,498 calculable records
4,502 records with no closed_date
Mean: 234.3 hours (~9.8 days)
Median: 14.9 hours
Minimum: −744 hours
Maximum: 14,984.8 hours (~624 days)


🚨 1. Negative resolution times

-744 hours is impossible if:

closed_date > created_date

So we have records where the closing timestamp occurs before the creation timestamp.

That's a clear data-quality issue we'll investigate.


2. The mean is misleading

Mean = 234 hours (~9.8 days)

Median = 14.9 hours

That's a huge difference.

The maximum is about 1,873 days, so a relatively small number of extremely long cases are pulling the mean upward.

This is actually a great analytical lesson:

For heavily skewed operational data, median and percentile statistics can be more representative than the mean.


3. We shouldn't simply delete the outliers

This is important.

A request taking thousands of hours might be:
genuinely unresolved for a long time,
a delayed administrative closure,
a data-entry issue,
or another legitimate edge case.

We'll investigate first.

In [19]:
# Check for negative resolution times
negative_resolution = df[df["resolution_time"] < 0]

print("Negative resolution records:", len(negative_resolution))

negative_resolution[
    ["unique_key", "created_date", "closed_date", "complaint_type", "agency"]
].head(10)


Negative resolution records: 182


,unique_key,created_date,closed_date,complaint_type,agency
9178,63591237,2025-01-01 19:07:52,2024-12-04 09:33:00,Sidewalk Condition,DOT
9644,63592286,2025-01-01 20:19:16,2024-12-04 09:33:00,Sidewalk Condition,DOT
28246,63622430,2025-01-03 09:07:00,2025-01-02 08:59:00,Street Light Condition,DOT
64138,63656893,2025-01-06 07:51:00,2025-01-04 02:00:00,Street Light Condition,DOT
109611,63694583,2025-01-09 11:05:00,2025-01-03 11:05:00,Street Light Condition,DOT
109635,63694823,2025-01-09 11:06:00,2025-01-03 11:06:00,Street Light Condition,DOT
109666,63692739,2025-01-09 11:07:00,2025-01-03 11:07:00,Street Light Condition,DOT
110499,63702502,2025-01-09 12:12:00,2025-01-06 12:12:00,Street Light Condition,DOT
110702,63697586,2025-01-09 12:26:00,2025-01-07 12:26:00,Street Light Condition,DOT
110748,63695071,2025-01-09 12:30:00,2025-01-07 22:30:00,Street Light Condition,DOT


In [21]:
negative_resolution[
    ["created_date", "closed_date", "resolution_time"]
].sort_values("resolution_time").head(10)

,created_date,closed_date,resolution_time
433307,2025-02-10 14:37:00,2025-01-10 14:36:00,-744.016667
9644,2025-01-01 20:19:16,2024-12-04 09:33:00,-682.771111
9178,2025-01-01 19:07:52,2024-12-04 09:33:00,-681.581111
198646,2025-01-16 21:11:00,2025-01-02 12:45:00,-344.433333
305079,2025-01-27 11:16:00,2025-01-15 11:14:00,-288.033333
305151,2025-01-27 11:22:00,2025-01-15 11:21:00,-288.016667
277006,2025-01-24 11:39:00,2025-01-13 11:37:00,-264.033333
405814,2025-02-07 11:20:00,2025-01-28 00:55:00,-250.416667
405793,2025-02-07 11:17:00,2025-01-28 01:00:00,-250.283333
405798,2025-02-07 11:18:00,2025-01-28 02:00:00,-249.300000


In [27]:
print("Negative resolution records:", df[df["resolution_time"] < 0].shape[0])
print("Percentage of negative resolution records:", (df[df["resolution_time"] < 0].shape[0] / df.shape[0]) * 100)

Negative resolution records: 182
Percentage of negative resolution records: 0.0364


In [28]:
print(
    df["resolution_time"].quantile(
        [0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

0.900     412.631944
0.950    1418.577500
0.990    4684.788844
0.995    5630.141944
0.999    9347.571417
Name: resolution_time, dtype: float64


For example:

created: 2025-01-01 19:07
closed:  2024-12-04 09:33

A request created in January 2025 cannot have been closed in December 2024. So these records should not contribute to resolution-time metrics.

But there's an important distinction:
We should not delete the entire complaint record just because its resolution time is invalid.

The complaint itself may still be perfectly useful for demand, geography, agency, complaint-type analysis

For records where:

closed_date < created_date

we'll eventually set their resolution time to missing (NaN), rather than deleting the records.






-------------------Data type and data-quality cleaning----------------------

In [63]:
df_clean = df.copy()

In [64]:
df_clean["created_date"] = pd.to_datetime(
    df_clean["created_date"],
    errors="coerce"
)

df_clean["closed_date"] = pd.to_datetime(
    df_clean["closed_date"],
    errors="coerce"
)

In [65]:
df_clean["resolution_time"] = (
    df_clean["closed_date"] - df_clean["created_date"]
).dt.total_seconds() / 3600

In [66]:
df_clean.loc[
    df_clean["resolution_time"] < 0,
    "resolution_time"
] = np.nan

In [67]:
print("Negative resolution times:",
      (df_clean["resolution_time"] < 0).sum())

print("Missing resolution times:",
      df_clean["resolution_time"].isna().sum())

Negative resolution times: 0
Missing resolution times: 4684


Duplicate records

In [68]:
print("Completely duplicated rows:",
      df_clean.duplicated().sum())

print("Duplicate unique_keys:",
      df_clean["unique_key"].duplicated().sum())

Completely duplicated rows: 0
Duplicate unique_keys: 0


Missing categorical values

In [69]:
# which agency is responsible for missing values
missing_cols = [
    "descriptor",
    "location_type",
    "incident_zip",
    "latitude",
    "longitude"
]

for col in missing_cols:
    print(f"\n===== {col} =====")
    print(
        df_clean[df_clean[col].isna()]
        ["agency"]
        .value_counts()
    )


===== descriptor =====
agency
NYPD     4936
DHS      3275
DOHMH      95
DOT         6
Name: count, dtype: int64

===== location_type =====
agency
DEP     22989
DOT     18165
DOB     11472
TLC       567
DSNY      363
NYPD        1
Name: count, dtype: int64

===== incident_zip =====
agency
DOT      1405
NYPD     1092
DSNY      591
DHS       562
TLC        66
DOHMH      32
DOB        28
HPD         8
DPR         7
DEP         6
DOE         5
DCWP        4
Name: count, dtype: int64

===== latitude =====
agency
DOT      3321
DSNY      845
NYPD      549
DEP       227
TLC        70
DHS        46
DOHMH      36
DPR        29
DOB        28
DCWP       25
HPD         8
DOE         5
Name: count, dtype: int64

===== longitude =====
agency
DOT      3321
DSNY      845
NYPD      549
DEP       227
TLC        70
DHS        46
DOHMH      36
DPR        29
DOB        28
DCWP       25
HPD         8
DOE         5
Name: count, dtype: int64


let's check whether we have suspicious placeholder values such as:

"Unspecified",
"UNKNOWN",
"OTHER",
blank strings, 
inconsistent capitalization

In [70]:
check_cols = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "status",
    "borough",
    "community_board",
    "open_data_channel_type"
]

for col in check_cols:
    print(f"\n===== {col} =====")
    print(
        df_clean[col]
        .value_counts(dropna=False)
        .tail(10)
    )


===== agency =====
agency
DEP      22992
DOB      11472
DOHMH     9386
DPR       6412
TLC       4229
DHS       3275
EDC       2682
DCWP      2051
DOE        360
OTI         38
Name: count, dtype: int64

===== agency_name =====
agency_name
Department of Environmental Protection          22992
Department of Buildings                         11472
Department of Health and Mental Hygiene          9386
Department of Parks and Recreation               6412
Taxi and Limousine Commission                    4229
Department of Homeless Services                  3275
Economic Development Corporation                 2682
Department of Consumer and Worker Protection     2051
Department of Education                           360
Office of Technology and Innovation                38
Name: count, dtype: int64

===== complaint_type =====
complaint_type
Unsanitary Animal Facility              2
Building Condition                      1
Unspecified                             1
Taxi Licensee Complaint  

--agency vs agency_name

They contain the same information.
We can eventually drop agency_name and keep the shorter agency code.
This is a defensible data-modeling decision: avoid redundant columns.


--status

"Unspecified" appears 38 times.
We should not convert it to "Closed" or another status.
Keep it as-is because it represents missing/unspecified information.


--borough

"Unspecified" appears 297 times.
Again, don't guess the borough.
Keep it as "Unspecified" for now.


--community_board

"Unspecified" appears 162 times.
Same principle.


--open_data_channel_type

"UNKNOWN" = 33,456
"OTHER" = 1
These are legitimate categorical values, so don't replace them with NaN.


--complaint_type, descriptor, location_type

The long list of categories isn't necessarily a data-quality problem.
Some legitimate categories occur only once.
Do not remove rare categories just because they're rare.

We haven't actually found any obvious duplicate/inconsistent spellings such as:

Closed
closed
CLOSED

or:

Unknown
UNKNOWN
unknown

So we don't need a blanket category-standardization step.

Check the numeric/geographic columns

In [71]:
numeric_cols = [
    "incident_zip",
    "latitude",
    "longitude",
    "resolution_time"
]

for col in numeric_cols:
    print(f"\n===== {col} =====")
    print(df_clean[col].describe())
    print("Missing:", df_clean[col].isna().sum())


===== incident_zip =====
count    496194.000000
mean      10776.967261
std         535.286952
min          83.000000
25%       10453.000000
50%       10472.000000
75%       11229.000000
max       78729.000000
Name: incident_zip, dtype: float64
Missing: 3806

===== latitude =====
count    494811.000000
mean         40.753909
std           0.093731
min          40.498949
25%          40.679378
50%          40.746893
75%          40.838826
max          40.912869
Name: latitude, dtype: float64
Missing: 5189

===== longitude =====
count    494811.000000
mean        -73.916404
std           0.072024
min         -74.254326
25%         -73.958355
50%         -73.917191
75%         -73.860168
max         -73.700723
Name: longitude, dtype: float64
Missing: 5189

===== resolution_time =====
count    495316.000000
mean        234.462958
std         845.080883
min           0.000000
25%           1.800000
50%          14.957917
75%          68.616806
max       14984.766111
Name: resolution_time, d

incident_zip has:

min = 83
max = 78729

Those aren't valid NYC ZIP codes. This is likely caused by bad/malformed source values, not something we should try to "fix" by guessing.

Also notice that incident_zip is currently float64 because it contains missing values. Since ZIP codes are identifiers rather than measurements, we'll eventually treat them as a categorical/geographic field rather than a numeric variable.

Investigate the suspicious ZIPs

In [72]:
df_clean.loc[
    (df_clean["incident_zip"] < 1000) |
    (df_clean["incident_zip"] > 11699),
    ["unique_key", "incident_zip", "borough", "community_board"]
].head(20)

,unique_key,incident_zip,borough,community_board
115331,63697124,35244.0,Unspecified,0 Unspecified
150338,63742563,83.0,MANHATTAN,64 MANHATTAN
221192,63814203,12345.0,MANHATTAN,10 MANHATTAN
260236,63852726,11767.0,Unspecified,0 Unspecified
294002,63891037,12345.0,MANHATTAN,10 MANHATTAN
314891,63916593,36602.0,Unspecified,0 Unspecified
341987,63946564,14614.0,Unspecified,0 Unspecified
361436,63961839,29601.0,Unspecified,0 Unspecified
389892,63998677,57117.0,Unspecified,0 Unspecified
434484,64042446,28262.0,Unspecified,0 Unspecified


Cleaning decision

For incident_zip:

Valid NYC ZIP → keep it.

Missing → keep missing.

Clearly invalid/non-NYC ZIP → set to NaN.

Don't delete the entire complaint record.

In [73]:
invalid_zip = (
    (df_clean["incident_zip"] < 1000) |
    (df_clean["incident_zip"] > 11699)
)

print("Invalid ZIP records:", invalid_zip.sum())
print("Percentage:", round(invalid_zip.mean() * 100, 4), "%")

Invalid ZIP records: 14
Percentage: 0.0028 %


In [74]:
df_clean.loc[
    invalid_zip,
    "incident_zip"
] = np.nan

In [75]:
print(
    "Invalid ZIPs remaining:",
    (
        (df_clean["incident_zip"] < 1000) |
        (df_clean["incident_zip"] > 11699)
    ).sum()
)

Invalid ZIPs remaining: 0


Check whether the date range is actually correct

In [76]:
print("Created date range:")
print("Minimum:", df_clean["created_date"].min())
print("Maximum:", df_clean["created_date"].max())

print("\nClosed date range:")
print("Minimum:", df_clean["closed_date"].min())
print("Maximum:", df_clean["closed_date"].max())

Created date range:
Minimum: 2025-01-01 00:00:12
Maximum: 2025-02-18 09:00:00

Closed date range:
Minimum: 2024-12-04 09:33:00
Maximum: 2026-09-29 00:00:00


Let's see exactly how many records have a closed_date after the current end of our sample's creation period.

In [77]:
future_closed = (
    df_clean["closed_date"] > df_clean["created_date"].max()
)

print("Closed after sample's latest created date:", future_closed.sum())


Closed after sample's latest created date: 55909


In [78]:
print("Latest created date:", df_clean["created_date"].max())
print("Latest closed date:", df_clean["closed_date"].max())

Latest created date: 2025-02-18 09:00:00
Latest closed date: 2026-09-29 00:00:00


Our dataset is:

Created: Jan 1, 2025 → Feb 18, 2025

Closed: Dec 4, 2024 → Sep 29, 2026

The closed_date range is broader because closure dates aren't restricted to the creation-date window.

The 182 impossible negative resolution durations have already been converted to NaN.

So no further date cleaning is needed.

Decide what to do with redundant columns

In [79]:
# Check if agency and agency_name are consistent
print(
    (df_clean["agency"].map(
        df_clean.groupby("agency")["agency_name"].first()
    ) == df_clean["agency_name"]).mean()
)

1.0


Remove the redundant column

In [80]:
df_clean.drop(columns=["agency_name"], inplace=True)

In [81]:
#check number of columns after dropping agency_name
df_clean.columns

Index(['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type',
       'descriptor', 'location_type', 'status', 'borough', 'incident_zip',
       'community_board', 'open_data_channel_type', 'latitude', 'longitude',
       'resolution_time'],
      dtype='str')

In [82]:
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate IDs:", df_clean["unique_key"].duplicated().sum())
print("Negative resolution times:",
      (df_clean["resolution_time"] < 0).sum())
print("Invalid ZIPs:",
      ((df_clean["incident_zip"] < 1000) |
       (df_clean["incident_zip"] > 11699)).sum())

Rows: 500000
Columns: 15
Duplicate rows: 0
Duplicate IDs: 0
Negative resolution times: 0
Invalid ZIPs: 0


In [84]:
df_clean.to_csv(
    "../data/processed/nyc_311_2025_clean.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


In [85]:
df_check = pd.read_csv(
    "../data/processed/nyc_311_2025_clean.csv"
)

print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))
print("File loaded successfully!")

Rows: 500000
Columns: 15
File loaded successfully!
